# <p style='font-family: https://fonts.google.com/share?selection.family=Signika+Negative:wght@300..700; background-color:#F3E5F5; font-weight:bold; color:#9C27B0; border:4px solid #9C27B0; border-radius:12px; box-shadow: 0px 4px 12px rgba(0, 0, 0, 0.2); padding:10px; text-align:center; transition: all 0.3s ease;'>🌟 LLM Powerd Resume - Career Recommendation System💎</p>

<div style="
  background-color: #F8EAF6; 
  border-left: 6px solid #8E24AA; 
  border-top: 6px solid #8E24AA; 
  border-radius: 12px; 
  padding: 16px 22px; 
  font-size: 16px; 
  font-family: 'Signika Negative', sans-serif; 
  color: #4A148C; 
  box-shadow: 0 4px 10px rgba(0, 0, 0, 0.15); 
  transition: transform 0.3s ease, box-shadow 0.3s ease;">
  
  <h4 style="
    font-size: 18px; 
    color: #7B1FA2; 
    font-weight: bold; 
    margin-bottom: 12px;">
    💠 Build Career Recommendation System
  </h4>

  <p style="
    margin: 0; 
    font-size: 15px; 
    line-height: 1.6; 
    color: #4A148C;">
  </p>
</div>

In [13]:
import os
import json
from dotenv import load_dotenv
from PyPDF2 import PdfReader

from typing import List
from pydantic import BaseModel, Field

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader

# <p style = 'font-family: https://fonts.google.com/share?selection.family=Signika+Negative:wght@300..700; background-color:#F3E5F5; font-weight:bold; color:#9C27B0; border:4px solid #9C27B0; border-radius:12px; box-shadow: 0px 4px 12px rgba(0, 0, 0, 0.2); padding:10px; text-align:center; transition: all 0.3s ease;'>🌟 Load API Key💎</p>

In [5]:
# LOAD API KEY & INITIALIZE GEMINI MODEL

load_dotenv()

# DEFINE GEMINI MODEL
llm = ChatGoogleGenerativeAI(model = "models/gemma-3-27b-it", temperature = 0.1)  # --> LOW TEMPERATURE TO INCREASE CONSISTENCY
llm

ChatGoogleGenerativeAI(profile={}, google_api_key=SecretStr('**********'), model='models/gemma-3-27b-it', temperature=0.1, client=<google.genai.client.Client object at 0x00000290467FFB00>, default_metadata=(), model_kwargs={})

# <p style = 'font-family: https://fonts.google.com/share?selection.family=Signika+Negative:wght@300..700; background-color:#F3E5F5; font-weight:bold; color:#9C27B0; border:4px solid #9C27B0; border-radius:12px; box-shadow: 0px 4px 12px rgba(0, 0, 0, 0.2); padding:10px; text-align:center; transition: all 0.3s ease;'>🌟 Build Schema💎</p>

In [ ]:
# CREATE SCHEMA

# FEATURE (INPUT SCHEMA)
class jobMatching(BaseModel):
    job_title: str = Field(description = "Predicted Job Title")
    probability: int = Field(description = "Match Probability Percentage (0 - 100)")
    reason: str = Field(description = "Short Explanation based on resume evidence")

# NUMBER OF MATCHING (OUTPUT SCHEMA)
class jobPrediction(BaseModel):
    jobs : List[jobMatching]

# DEFINE PARSER
parser = PydanticOutputParser(pydantic_object = jobPrediction)

parser

PydanticOutputParser(pydantic_object=<class '__main__.jobPrediction'>)

# <p style = 'font-family: https://fonts.google.com/share?selection.family=Signika+Negative:wght@300..700; background-color:#F3E5F5; font-weight:bold; color:#9C27B0; border:4px solid #9C27B0; border-radius:12px; box-shadow: 0px 4px 12px rgba(0, 0, 0, 0.2); padding:10px; text-align:center; transition: all 0.3s ease;'>🌟 Build Prompt💎</p>

In [12]:
# CREATE PROMPT

PROMPT = PromptTemplate(template = """You are an AI Career Analyst system. Analyze the following resume text and determine list of most suitable job positions (job titles) 
                            for this candidate based on their skills, experience, and education.
                            Rules:
                            - Return multiple job predictions.
                            - Probability must be between 1-100.
                            - Base reasoning strictly on resume evidence.
                            - Do NOT include any text outside the required format.
                            Resume: {resume_text}        
                            {format_instructions}""",
                        
                        input_variables=["resume_text"],
                        partial_variables={"format_instructions": parser.get_format_instructions()})

# <p style = 'font-family: https://fonts.google.com/share?selection.family=Signika+Negative:wght@300..700; background-color:#F3E5F5; font-weight:bold; color:#9C27B0; border:4px solid #9C27B0; border-radius:12px; box-shadow: 0px 4px 12px rgba(0, 0, 0, 0.2); padding:10px; text-align:center; transition: all 0.3s ease;'>🌟 Data Ingestion : Accept PDF File 💎</p>

In [14]:
# PDF LOADER FUNCTION

def load_pdf(pdf_path : str):

    # LOAD PDF 
    loader = PyPDFLoader(file_path = pdf_path)
    pages = loader.load()

    # RETURN MERGED PAGES
    return "\n".join([page.page_content for page in pages])

# <p style = 'font-family: https://fonts.google.com/share?selection.family=Signika+Negative:wght@300..700; background-color:#F3E5F5; font-weight:bold; color:#9C27B0; border:4px solid #9C27B0; border-radius:12px; box-shadow: 0px 4px 12px rgba(0, 0, 0, 0.2); padding:10px; text-align:center; transition: all 0.3s ease;'>🌟 Pipeline💎</p>

In [ ]:
# FUNCTION TO DETECT SUITABLE JOBS

def detect_jobs(pdf_path : str):

    # RUN LOAD PDF FUNCTION
    resume_text = load_pdf(pdf_path)

    # INSERT LOADED CV INTO PROMPT
    formatted_prompt = PROMPT.format(resume_text = resume_text)

    # RUN GEMINI MODEL 
    results = llm.invoke(formatted_prompt)

    # PARSE RESULT INTO JSON FORMAT
    parsed = parser.parse(results.content)

    return parsed

In [18]:
# FUNCTION TO DISPLAY OUTPUT AS JSON FORMAT

def display_output(results : jobPrediction):

    # SORT JOBS BY PROBABILITY AS DESCENDING
    sorted_jobs = sorted(results.jobs, key = lambda x : x.probability, reverse = True)

    # DISPLAY OUTPUT
    for job in sorted_jobs:
        print(f"{job.job_title} - {job.probability}%")
        print(f"Reason: {job.reason}")
        print("-" * 40)

# <p style = 'font-family: https://fonts.google.com/share?selection.family=Signika+Negative:wght@300..700; background-color:#F3E5F5; font-weight:bold; color:#9C27B0; border:4px solid #9C27B0; border-radius:12px; box-shadow: 0px 4px 12px rgba(0, 0, 0, 0.2); padding:10px; text-align:center; transition: all 0.3s ease;'>🌟 Test Model💎</p>

In [19]:
# TEST MODEL

PDF_PATH = r"cv_computer-vision.pdf"

result = detect_jobs(pdf_path = PDF_PATH)
display_output(results = result)

Machine Learning Engineer - 95%
Reason: Extensive experience in designing, developing, and deploying ML/DL models using TensorFlow, PyTorch, and other relevant frameworks. Proven project experience with model training, evaluation, and deployment (e.g., fraud detection, recommendation systems).
----------------------------------------
AI Engineer - 92%
Reason: Participation in AI Engineer programs and completion of advanced training in ML/DL. Demonstrated ability to build and optimize predictive models using modern neural network architectures.
----------------------------------------
Computer Vision Engineer - 88%
Reason: Strong project focused on computer vision (Satellite-Based Mine Detection) utilizing Faster R-CNN and YOLOv8, demonstrating expertise in image processing and object detection.
----------------------------------------
Data Scientist - 85%
Reason: Proficiency in data analysis, preprocessing, and visualization using tools like Pandas, NumPy, and Matplotlib. Experience wi